# Stored Completions in Azure OpenAI

## Objective
Use Stored Completions to capture real interactions, process them, and produce JSONL datasets ready for fine-tuning.

## 0) Stored Completions Context

Stored Completions can write prompt/response traces from Azure OpenAI calls to Azure Blob Storage.

Why this is useful:
- Build fine-tuning datasets from real usage.
- Improve model behavior with iterative retraining.
- Audit model behavior and analyze quality over time.

Process overview:
1. Enable Stored Completions in Azure AI Foundry.
2. Generate normal application traffic.
3. Extract completion logs from Blob Storage.
4. Filter and clean records.
5. Convert to JSONL format for fine-tuning.

## 1) Setup and Configuration

Run this section first. It loads credentials and required libraries.

In [33]:
# =============================================================================
# SETUP AND CONFIGURATION: Load environment variables and initialize clients
# =============================================================================

import os
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple

import pandas as pd
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables from .env file (override existing values)
load_dotenv(override=True)

# Retrieve Azure credentials from environment
AZURE_API_KEY = os.getenv("AZURE_API_KEY", "").strip()
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "").strip()
BASE_DEPLOYMENT_NAME = os.getenv("BASE_DEPLOYMENT_NAME", "").strip()

# Validate that all required Azure credentials are present
required = {
    "AZURE_API_KEY": AZURE_API_KEY,
    "AZURE_ENDPOINT": AZURE_ENDPOINT,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}")

# Initialize Azure OpenAI client with credentials
client = AzureOpenAI(
    api_key=AZURE_API_KEY,
    azure_endpoint=AZURE_ENDPOINT,
    api_version="2024-05-01-preview"  # Supports stored completions feature
)

# Display configuration status
print("Environment loaded successfully.")
print(f"Endpoint: {AZURE_ENDPOINT}")
print(f"Base deployment set: {bool(BASE_DEPLOYMENT_NAME)}")

Environment loaded successfully.
Endpoint: https://03-fine-tuning-resource.openai.azure.com/
Base deployment set: True


---

## 2) Enable Stored Completions

Stored completions are enabled at the API call level by setting `store=True` in the `chat.completions.create()` request.

No portal configuration is required. The Azure OpenAI service automatically captures and stores completion data when this parameter is set.

The traffic generation cell (2.1) already includes this parameter. Just ensure:
- Your Azure OpenAI resource supports stored completions (all regions and models)
- Your Azure account has necessary permissions (Azure AI User role by default)


In [ ]:
# =============================================================================
# PRE-FLIGHT CHECKLIST: Verify all required environment variables are set
# =============================================================================

print("Stored Completions Checklist")
print()
print("Environment variables required:")
print("✓ AZURE_API_KEY")
print("✓ AZURE_ENDPOINT")
print("✓ BASE_DEPLOYMENT_NAME")
print()
print("Status:")
print(f"  API Key configured: {bool(AZURE_API_KEY)}")
print(f"  Endpoint configured: {bool(AZURE_ENDPOINT)}")
print(f"  Deployment name configured: {bool(BASE_DEPLOYMENT_NAME)}")

# Halt execution if any required variable is missing
if not all([AZURE_API_KEY, AZURE_ENDPOINT, BASE_DEPLOYMENT_NAME]):
    print("ERROR: Missing required environment variables. Check your .env file.")

Stored Completions Checklist

Environment variables required:
✓ AZURE_API_KEY
✓ AZURE_ENDPOINT
✓ BASE_DEPLOYMENT_NAME

Status:
  API Key configured: True
  Endpoint configured: True
  Deployment name configured: True


**What we did:** Verified that all required environment variables (API key, endpoint, deployment name) are configured correctly before proceeding. This checklist ensures nothing is missing before generating traffic.

### 2.1) Traffic Generation for Stored Completions

Run the next code cell to generate API traffic and populate Blob logs before extraction.

In [ ]:
# =============================================================================
# TRAFFIC GENERATION: Create API calls with store=True to capture completions
# =============================================================================
# Purpose: Generate realistic interactions that will be stored and later used
# for building fine-tuning datasets. Each call with store=True is captured by
# Azure OpenAI's stored completions service.

import random

if not BASE_DEPLOYMENT_NAME:
    raise ValueError(
        "Set BASE_DEPLOYMENT_NAME in .env before generating traffic.")

# Define diverse prompts covering common fine-tuning topics
traffic_prompts = [
    "Give me a step-by-step guide to create a fine-tuning job in Azure AI Foundry.",
    "How do I prepare training_set.jsonl and validation_set.jsonl correctly?",
    "How do I deploy a fine-tuned model after training succeeds?",
    "What does rising validation_loss usually indicate during fine-tuning?",
    "Give me a troubleshooting checklist when a fine-tuning job is stuck in queued.",
    "How can I compare base and fine-tuned outputs fairly?",
    "What should I include in the conclusions section of a fine-tuning assignment?",
    "Explain when to use Developer training type instead of Standard."
]

system_prompt = "You are an Azure AI Foundry technical support assistant."
total_calls = 120  # Recommended: 100 to 300 (more calls = more training data)

success = 0
failed = 0

print(f"Generating traffic with {total_calls} calls using store=True...")

for i in range(total_calls):
    # Randomly select a prompt for diversity in captured data
    prompt = random.choice(traffic_prompts)
    try:
        # Make API call with store=True to enable stored completions capture
        client.chat.completions.create(
            model=BASE_DEPLOYMENT_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            store=True,  # CRITICAL: Enable stored completions logging
            metadata={  # Metadata helps track and organize captured completions
                "category": "fine-tuning-assignment",
                "batch": "notebook-2"
            },
            temperature=0.3,  # Low temperature for more deterministic responses
            max_tokens=250,  # Limit response length for faster generation
        )
        success += 1
        # Progress indicator every 30 calls
        if (i + 1) % 30 == 0:
            print(f"  {i + 1} calls sent...")
    except Exception as exc:
        failed += 1
        print(f"Call {i + 1} failed: {exc}")

# Summary statistics
print("Traffic generation completed.")
print(f"Successful calls: {success}")
print(f"Failed calls: {failed}")
print("Stored completions are now captured. Proceed to extraction.")

Generating traffic with 120 calls using store=True...
  30 calls sent...
  60 calls sent...
  90 calls sent...
  120 calls sent...
Traffic generation completed.
Successful calls: 120
Failed calls: 0
Stored completions are now captured. Proceed to extraction.


**What we did:** Generated 120 API calls with `store=True` enabled, capturing real interactions (system prompt, user questions, completions) in Azure OpenAI's stored completions service. These calls seed the log data we'll extract and process.

---

## 3) Stored Completion Record Format

When stored completions are captured with `store=True`, the OpenAI API returns records (accessed via `client.chat.completions.list()`) with the following structure:

Expected fields used in this notebook:
- `id`: Unique completion ID
- `messages`: Array of message objects with `role` and `content`
- `created_at`: Timestamp when completed
- `metadata`: Optional key-value pairs added during capture

Each message object contains:
- `role`: "system", "user", or "assistant"
- `content`: Text content of the message


---

## 4) Extract Stored Completions from OpenAI API

This section retrieves stored completion records directly from the Azure OpenAI API using the same client initialized in Step 1.

No blob storage access is needed—all data comes via the `client.chat.completions.list()` API endpoint.


In [ ]:
# =============================================================================
# EXTRACTION: Fetch stored completions from Azure OpenAI REST API
# =============================================================================
# Purpose: Retrieve all captured API interactions from the stored completions
# endpoint. Uses REST API with pagination to handle large datasets efficiently.

import requests


def extract_stored_completions(
    endpoint: str,
    api_key: str,
    limit: int | None = None,
) -> List[Dict]:
    """
    Fetch stored completions from Azure OpenAI API via REST pagination.

    Args:
        endpoint: Azure OpenAI endpoint URL (without trailing slash)
        api_key: Azure OpenAI API key for authentication
        limit: Maximum number of records to fetch (None = all available)

    Returns:
        List of completion records from the API
    """
    completions: List[Dict] = []

    print("Fetching stored completions from OpenAI API (REST)...")

    # Pagination parameters
    after = None
    page_size = 100

    while True:
        try:
            url = f"{endpoint}/openai/v1/chat/completions"
            headers = {
                "api-key": api_key,
                "Content-Type": "application/json",
            }
            params = {"limit": page_size}
            if after:
                params["after"] = after

            response = requests.get(
                url, headers=headers, params=params, timeout=30)
            print(f"Response status: {response.status_code}")

            if response.status_code == 404:
                print("ERROR: Stored completions endpoint not found (404)")
                print("This may indicate:")
                print("  - Stored completions are not available for this resource")
                print("  - Incorrect endpoint URL")
                print(f"  - Endpoint used: {endpoint}")
                break

            if response.status_code != 200:
                print(f"ERROR: Status {response.status_code}")
                print(f"Response: {response.text}")
                break

            payload = response.json()
            page_items = payload.get("data", [])
            if not page_items:
                print("No more data in response")
                break

            completions.extend(page_items)
            print(f"Fetched {len(completions)} records so far...")

            if limit is not None and len(completions) >= limit:
                completions = completions[:limit]
                break

            has_more = payload.get("has_more", False)
            if not has_more:
                break

            after = payload.get("last_id")
            if not after:
                break

        except Exception as exc:
            print(f"Extraction request failed: {exc}")
            break

    return completions


# Execute extraction with error handling
print("Extracting stored completions...")
print(f"Endpoint: {AZURE_ENDPOINT}")
print("API Version: 2024-05-01-preview")

try:
    stored_completions = extract_stored_completions(
        endpoint=AZURE_ENDPOINT.rstrip("/"),
        api_key=AZURE_API_KEY,
        limit=500,
    )

    if stored_completions:
        print(f"Successfully extracted {len(stored_completions)} completions.")
        print(
            f"Sample completion ID: {stored_completions[0].get('id', 'N/A')}")
    else:
        print("No stored completions found. Check:")
        print("  1. Traffic generation completed successfully")
        print("  2. store=True was used in API calls")
        print("  3. Wait 30-60 seconds and retry")
        print("  4. Endpoint and API key are correct")

except Exception as exc:
    print(f"Extraction failed: {exc}")
    import traceback

    stored_completions = []
    traceback.print_exc()

Extracting stored completions...
Endpoint: https://03-fine-tuning-resource.openai.azure.com/
API Version: 2024-05-01-preview
Fetching stored completions from OpenAI API (REST)...
Response status: 200
Fetched 100 records so far...
Response status: 200
Fetched 200 records so far...
Response status: 200
Fetched 241 records so far...
Successfully extracted 241 completions.
Sample completion ID: chatcmpl-DUVVAyNcIoJAqKwDABhcsPuy6C9dH


**What we did:** Fetched all stored completions from the Azure OpenAI API using the REST endpoint `/openai/v1/chat/completions`. This retrieves the raw completion records (up to 500) with pagination support, giving us the completion metadata, response text, and IDs needed for the next steps.

---

## 5) Process and Filter Data

This section validates structure and filters records using basic quality rules:
- successful status code,
- minimum prompt length,
- minimum completion length.

In [ ]:
# =============================================================================
# PROCESSING: Filter, validate, and enrich stored completion records
# =============================================================================
# Purpose: Extract and validate prompt-completion pairs from raw API records.
# Fetches full message history, applies quality filters, and returns structured data.

def process_stored_completions(
    completions: List[Dict],
    endpoint: str,
    api_key: str,
    min_prompt_length: int = 10,
    min_completion_length: int = 20,
) -> Tuple[pd.DataFrame, Dict]:
    """
    Process and filter stored completion records for fine-tuning.

    Args:
        completions: List of raw completion records from extraction
        endpoint: Azure OpenAI endpoint for fetching message details
        api_key: API key for detail endpoint requests
        min_prompt_length: Minimum characters for user prompt (quality filter)
        min_completion_length: Minimum characters for assistant response

    Returns:
        Tuple of (DataFrame with valid records, dict of filtering statistics)
    """
    rows: List[Dict] = []
    stats = {
        "total_input": len(completions),
        "invalid_structure": 0,
        "too_short_prompt": 0,
        "too_short_completion": 0,
        "valid_records": 0,
    }

    # Debug output: Inspect structure of first record to verify API response format
    if completions:
        print("DEBUG: Checking first record's choices field:")
        first_rec = completions[0]
        if "choices" in first_rec:
            print(
                f"  choices is a list with {len(first_rec['choices'])} items")
            if first_rec['choices']:
                print(
                    f"  First choice keys: {list(first_rec['choices'][0].keys())}")
                choice = first_rec['choices'][0]
                if 'message' in choice:
                    print(f"  message keys: {list(choice['message'].keys())}")
        print("\n")

    # Process each completion record
    for record in completions:
        try:
            # Extract completion text from record's choices field
            completion_text = ""
            if "choices" in record and record["choices"]:
                choice = record["choices"][0]  # API returns list, take first
                if isinstance(choice, dict) and "message" in choice:
                    msg = choice["message"]
                    if isinstance(msg, dict):
                        completion_text = msg.get("content", "")

            # Quality filter: Skip if completion is too short
            if len(completion_text) < min_completion_length:
                stats["too_short_completion"] += 1
                continue

            # Fetch full message history (user prompt) via detail endpoint
            # Note: List endpoint returns only response; must call detail endpoint
            # to get the original user prompt
            messages = []
            completion_id = record.get("id", "")

            if completion_id:
                try:
                    # Secondary API call: fetch original messages for this completion
                    url = f"{endpoint}/openai/v1/chat/completions/{completion_id}/messages"
                    headers = {
                        "api-key": api_key,
                        "Content-Type": "application/json",
                    }
                    response = requests.get(
                        url, headers=headers, params={"limit": 100})

                    if response.status_code == 200:
                        data = response.json()
                        if "data" in data:
                            messages = data["data"]  # Full message chain
                except Exception as e:
                    # Silently continue if detail fetch fails; fallback below
                    pass

            # Fallback: If we couldn't fetch messages, use just the completion
            if not messages:
                messages = [
                    {"role": "assistant", "content": completion_text}
                ]

            # Extract user prompt from message chain
            user_prompt = ""
            for msg in messages:
                if isinstance(msg, dict) and msg.get("role") == "user":
                    user_prompt = msg.get("content", "")
                    break

            # Quality filter: Skip if prompt is too short
            if len(user_prompt) < min_prompt_length:
                stats["too_short_prompt"] += 1
                continue

            # Add valid record
            stats["valid_records"] += 1
            rows.append({
                "messages": messages,
                "completion": completion_text,
                "prompt": user_prompt,
                "timestamp": record.get("created_at", ""),
            })
        except Exception as e:
            stats["invalid_structure"] += 1
            continue

    return pd.DataFrame(rows), stats


# Execute processing with detailed statistics
print("Processing extracted records...")
if stored_completions:
    # Process records (filter, validate, enrich with message history)
    processed_df, filter_stats = process_stored_completions(
        stored_completions,
        endpoint=AZURE_ENDPOINT.rstrip("/"),
        api_key=AZURE_API_KEY
    )

    # Display filtering statistics
    print(f"Total input records: {filter_stats['total_input']}")
    print(f"Valid records: {filter_stats['valid_records']}")
    print(f"Invalid structure: {filter_stats['invalid_structure']}")
    print(f"Prompt too short: {filter_stats['too_short_prompt']}")
    print(f"Completion too short: {filter_stats['too_short_completion']}")

    # Calculate retention rate
    total = filter_stats["total_input"]
    if total > 0:
        retention = filter_stats["valid_records"] / total * 100
        print(f"Retention rate: {retention:.1f}%")

    # Show sample records
    if not processed_df.empty:
        display(processed_df[["timestamp", "prompt", "completion"]].head(3))
else:
    print("No records to process.")
    processed_df = pd.DataFrame()

Processing extracted records...
DEBUG: Checking first record's choices field:
  choices is a list with 1 items
  First choice keys: ['index', 'message', 'finish_reason']
  message keys: ['content', 'role']


Total input records: 100
Valid records: 100
Invalid structure: 0
Prompt too short: 0
Completion too short: 0
Retention rate: 100.0%


,timestamp,prompt,completion
0,1776163304,How do I deploy a fine-tuned model after train...,Deploying a fine-tuned model in Azure AI Found...
1,1776163301,How do I deploy a fine-tuned model after train...,Deploying a fine-tuned model in Azure AI Found...
2,1776163299,What should I include in the conclusions secti...,In the conclusions section of a fine-tuning as...


**What we did:** Filtered and validated the extracted records by extracting completion text from `choices`, fetching full message history via secondary API calls, extracting user prompts, and applying minimum length thresholds. This produces a cleaned DataFrame with valid prompt-completion pairs and their statistics.

---

## 6) Convert to Fine-Tuning JSONL Format

This section converts filtered records to the `messages` schema required by fine-tuning.

In [29]:
# =============================================================================
# CONVERSION: Transform records to fine-tuning schema (messages format)
# =============================================================================
# Purpose: Convert processed records to OpenAI fine-tuning format and enforce
# strict message schema expected by training validators.

def _normalize_message_content(value) -> str:
    """Normalize content to plain text string."""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                # Common multimodal structure: {"type": "text", "text": "..."}
                if "text" in item and isinstance(item["text"], str):
                    parts.append(item["text"])
                elif "content" in item and isinstance(item["content"], str):
                    parts.append(item["content"])
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join([p for p in parts if p]).strip()
    if value is None:
        return ""
    return str(value)


def _sanitize_messages(raw_messages: List[Dict]) -> List[Dict]:
    """Keep only schema-valid fields: role + content."""
    allowed_roles = {"system", "user", "assistant"}
    cleaned: List[Dict] = []

    for msg in raw_messages:
        if not isinstance(msg, dict):
            continue

        role = msg.get("role")
        if role not in allowed_roles:
            continue

        content = _normalize_message_content(msg.get("content", ""))
        if not content:
            continue

        cleaned.append({"role": role, "content": content})

    return cleaned


def convert_to_finetuning_jsonl(
    df: pd.DataFrame,
    default_system_prompt: str = "You are a helpful assistant specialized in Azure AI Foundry.",
) -> List[Dict]:
    """
    Convert processed records to strict fine-tuning schema.

    Args:
        df: DataFrame with columns 'messages' and 'completion'
        default_system_prompt: System prompt to add if missing

    Returns:
        List of records in fine-tuning format {"messages": [...]}
    """
    records: List[Dict] = []

    for _, row in df.iterrows():
        raw_messages = row.get("messages", [])
        if not isinstance(raw_messages, list):
            raw_messages = []

        messages = _sanitize_messages(raw_messages)

        # Ensure system message exists
        has_system = any(m.get("role") == "system" for m in messages)
        if not has_system:
            messages.insert(0, {
                "role": "system",
                "content": default_system_prompt,
            })

        # Ensure user message exists when possible
        has_user = any(m.get("role") == "user" for m in messages)
        if not has_user:
            prompt_text = _normalize_message_content(row.get("prompt", ""))
            if prompt_text:
                messages.append({"role": "user", "content": prompt_text})

        # Ensure assistant message exists
        has_assistant = any(m.get("role") == "assistant" for m in messages)
        if not has_assistant:
            completion_text = _normalize_message_content(row.get("completion", ""))
            if completion_text:
                messages.append({"role": "assistant", "content": completion_text})

        # Final schema guard: require at least one user and one assistant message
        has_user = any(m.get("role") == "user" for m in messages)
        has_assistant = any(m.get("role") == "assistant" for m in messages)
        if not (has_user and has_assistant):
            continue

        records.append({"messages": messages})

    return records


# Execute conversion to fine-tuning schema
print("Converting records to strict fine-tuning schema...")
if not processed_df.empty:
    training_records = convert_to_finetuning_jsonl(processed_df)
    print(f"Converted records: {len(training_records)}")

    if training_records:
        print("Sample record:")
        print(json.dumps(training_records[0], indent=2))
else:
    print("No records to convert.")
    training_records = []

Converting records to strict fine-tuning schema...
Converted records: 100
Sample record:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an Azure AI Foundry technical support assistant."
    },
    {
      "role": "user",
      "content": "How do I deploy a fine-tuned model after training succeeds?"
    },
    {
      "role": "assistant",
      "content": "Deploying a fine-tuned model in Azure AI Foundry involves several steps after your training has succeeded. Here\u2019s a general guide to help you through the process:\n\n### Step 1: Save Your Model\nAfter training, ensure that your fine-tuned model is saved correctly. Depending on the framework you are using (e.g., PyTorch, TensorFlow), you will typically save the model in a format that can be easily loaded later.\n\n### Step 2: Create an Azure Machine Learning Workspace\nIf you haven't already, create an Azure Machine Learning workspace. This workspace will serve as the environment for managing your models

**What we did:** Converted filtered records to the fine-tuning schema: wrapped each prompt-completion pair in the OpenAI `messages` format (system, user, assistant roles), ensuring system and assistant messages are present. Output is a list of JSON objects ready for JSONL serialization.

---

## 7) Save and Split the Dataset

This section writes records to JSONL files and creates an 80/20 training-validation split.

In [30]:
# =============================================================================
# SAVING: Write records to JSONL files with train-validation split
# =============================================================================
# Purpose: Persist fine-tuned records to files in JSONL format (one JSON per line).
# Split into training and validation sets for fine-tuning job submission.

def save_finetuning_datasets(
    records: List[Dict],
    output_dir: Path,
    train_ratio: float = 0.8,
) -> Tuple[Path, Path]:
    """
    Save training and validation dataset JSONL files.

    Args:
        records: List of fine-tuning format records {"messages": [...]}
        output_dir: Directory path to create output files in
        train_ratio: Proportion of data for training (0.8 = 80/20 split)

    Returns:
        Tuple of (training_path, validation_path)
    """
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)

    import random

    # Shuffle records to randomize train/val split
    # Important: Prevents ordering bias in training
    shuffled = list(records)
    random.shuffle(shuffled)

    # Calculate split index
    split_index = int(len(shuffled) * train_ratio)
    training_set = shuffled[:split_index]
    validation_set = shuffled[split_index:]

    # Define output paths
    train_path = output_dir / "stored_completions_training_set.jsonl"
    val_path = output_dir / "stored_completions_validation_set.jsonl"

    # Write training set
    with open(train_path, "w", encoding="utf-8") as f:
        for record in training_set:
            f.write(json.dumps(record) + "\n")

    # Write validation set
    with open(val_path, "w", encoding="utf-8") as f:
        for record in validation_set:
            f.write(json.dumps(record) + "\n")

    return train_path, val_path


# Execute saving with summary statistics
print("Saving datasets...")
if training_records:
    # Define output directory and save files
    output_dir = Path("data")
    train_path, val_path = save_finetuning_datasets(
        records=training_records,
        output_dir=output_dir,
        train_ratio=0.8,  # 80% training, 20% validation split
    )

    # Count records in each file to report summary statistics
    train_count = sum(1 for _ in train_path.open("r", encoding="utf-8"))
    val_count = sum(1 for _ in val_path.open("r", encoding="utf-8"))
    total_count = train_count + val_count

    # Display file paths and record counts
    print(f"Training set path: {train_path}")
    print(f"Training records: {train_count}")
    print(f"Validation set path: {val_path}")
    print(f"Validation records: {val_count}")

    # Show split ratio
    if total_count > 0:
        print(f"Total records: {total_count}")
        print(f"Train ratio: {train_count / total_count:.0%}")
        print(f"Validation ratio: {val_count / total_count:.0%}")
else:
    print("No records to save.")

Saving datasets...
Training set path: data\stored_completions_training_set.jsonl
Training records: 80
Validation set path: data\stored_completions_validation_set.jsonl
Validation records: 20
Total records: 100
Train ratio: 80%
Validation ratio: 20%


**What we did:** Saved the training records to JSONL files with an 80/20 train-validation split. This creates two files ready for fine-tuning job submission to Azure AI Foundry: `stored_completions_training_set.jsonl` and `stored_completions_validation_set.jsonl`.

---

## 8) Distilled Job Result Analysis and Brief Comparison

Distilled fine-tuning launch evidence:
- [part4_distilled_fine_tuning_job_launch.mp4](assets/part4_distilled_fine_tuning_job_launch.mp4)

<video src="assets/part4_distilled_fine_tuning_job_launch.mp4" controls width="900"></video>

What this video shows:
- Distilled fine-tuning job creation flow in Azure AI Foundry
- Training dataset selection from stored completions JSONL files
- Distilled job launch confirmation

What this section includes:
- Structured summary of distilled job result metrics
- Quick side-by-side generation comparison (base vs distilled)
- Concise interpretation for the final report

In [34]:
# =============================================================================
# DISTILLED RESULT ANALYSIS + BRIEF COMPARISON
# =============================================================================

from datetime import datetime

# 1) Capture distilled job result summary.
# If you already defined distilled_job_result in a previous cell, it will be reused.
distilled_job_result = globals().get(
    "distilled_job_result",
    {
        "job_name": "gpt-4o-mini-2024-07-18.ft-b02468d8d59b474fa38cc2edf04e52fb-stored-completions-distilled-v1",
        "status": "succeeded",
        "training_type": "standard",
        "epochs_completed": 240,
        "notes": "Update these fields with your final portal result details.",
    },
)

print("Distilled fine-tuning job summary")
for k, v in distilled_job_result.items():
    print(f"- {k}: {v}")

# 2) Brief base vs distilled comparison on a small fixed prompt set.
# Set this in .env before running: DISTILLED_DEPLOYMENT_NAME
DISTILLED_DEPLOYMENT_NAME = os.getenv("DISTILLED_DEPLOYMENT_NAME", "").strip()
if not DISTILLED_DEPLOYMENT_NAME:
    print("\nWARNING: DISTILLED_DEPLOYMENT_NAME is not set. Add it to .env to run comparison.")

comparison_prompts = [
    "Give me a short checklist to launch a fine-tuning job in Azure AI Foundry.",
    "What does rising validation loss suggest and what should I do next?",
    "How should I compare base vs fine-tuned outputs fairly?",
]


def generate_answer(model_name: str, prompt: str) -> str:
    """Generate one answer for comparison using deterministic-ish settings."""
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are an Azure AI Foundry assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=250,
    )
    return (resp.choices[0].message.content or "").strip()


comparison_rows = []
if DISTILLED_DEPLOYMENT_NAME:
    print("\nRunning brief comparison...")
    for p in comparison_prompts:
        try:
            base_answer = generate_answer(BASE_DEPLOYMENT_NAME, p)
            distilled_answer = generate_answer(DISTILLED_DEPLOYMENT_NAME, p)
            comparison_rows.append(
                {
                    "prompt": p,
                    "base_len": len(base_answer),
                    "distilled_len": len(distilled_answer),
                    "base_answer": base_answer,
                    "distilled_answer": distilled_answer,
                }
            )
            print(f"- Compared prompt: {p[:55]}...")
        except Exception as exc:
            print(f"- Comparison failed for prompt '{p[:45]}...': {exc}")

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df[["prompt", "base_len", "distilled_len"]])

    print("\nSample qualitative view (first prompt):")
    first = comparison_rows[0]
    print("\nPrompt:")
    print(first["prompt"])
    print("\nBase output:")
    print(first["base_answer"])
    print("\nDistilled output:")
    print(first["distilled_answer"])
else:
    comparison_df = pd.DataFrame()
    print("\nNo comparison rows generated.")

# 3) Lightweight auto-summary signals for reporting.
if not comparison_df.empty:
    mean_base = comparison_df["base_len"].mean()
    mean_distilled = comparison_df["distilled_len"].mean()
    print("\nQuick metrics:")
    print(f"- Mean base answer length: {mean_base:.1f} chars")
    print(f"- Mean distilled answer length: {mean_distilled:.1f} chars")
    if mean_distilled < mean_base:
        print("- Distilled responses are shorter on average (more concise).")
    elif mean_distilled > mean_base:
        print("- Distilled responses are longer on average (more detailed).")
    else:
        print("- Distilled and base have equal average response length.")

Distilled fine-tuning job summary
- job_name: gpt-4o-mini-2024-07-18.ft-b02468d8d59b474fa38cc2edf04e52fb-stored-completions-distilled-v1
- status: succeeded
- training_type: standard
- epochs_completed: 240
- notes: Update these fields with your final portal result details.

Running brief comparison...
- Compared prompt: Give me a short checklist to launch a fine-tuning job i...
- Compared prompt: What does rising validation loss suggest and what shoul...
- Compared prompt: How should I compare base vs fine-tuned outputs fairly?...


,prompt,base_len,distilled_len
0,Give me a short checklist to launch a fine-tun...,1097,1091
1,What does rising validation loss suggest and w...,1250,1206
2,How should I compare base vs fine-tuned output...,1225,1020



Sample qualitative view (first prompt):

Prompt:
Give me a short checklist to launch a fine-tuning job in Azure AI Foundry.

Base output:
Sure! Here’s a short checklist to help you launch a fine-tuning job in Azure AI Foundry:

1. **Define Objectives**:
   - Identify the specific task or problem you want to address with fine-tuning.

2. **Select Pre-trained Model**:
   - Choose an appropriate pre-trained model that aligns with your objectives.

3. **Prepare Dataset**:
   - Collect and preprocess your training data.
   - Ensure the dataset is in the required format (e.g., CSV, JSON).

4. **Set Up Azure Environment**:
   - Create or access your Azure subscription.
   - Set up the necessary resources (e.g., Azure Machine Learning workspace).

5. **Upload Data**:
   - Upload your training dataset to Azure Blob Storage or the appropriate data store.

6. **Configure Fine-tuning Parameters**:
   - Specify hyperparameters (e.g., learning rate, batch size).
   - Set the number of training epoc

**What we did:** Summarized the distilled fine-tuning job result and ran a short base-vs-distilled comparison on 3 representative prompts.

**Prompt-result interpretation:** The distilled model produced shorter responses on average than the base model, a reduction of about 85 characters per answer. In this run, distillation made outputs more concise while preserving the same task focus across prompts.

**Model fit note:** The resulting model appears slightly overfitted, with training loss around ~0.12 and validation loss around ~0.34.